In [ ]:
import pandas as pd, io

# ============================================================
# PAS 0. Citirea datelor din fișierul local
# ============================================================
df = pd.read_csv("lab5.csv", dtype={'caem': str})

df['caem'] = df['caem'].str.zfill(4)
df['diviziune'] = df['caem'].str[:2].astype(int)

# ============================================================
# PAS 1. Secțiune din diviziune; coduri invalide; structură
# ============================================================
rng = [('A',1,3),('B',5,9),('C',10,33),('D',35,35),('E',36,39),('F',41,43),
       ('G',45,47),('H',49,53),('I',55,56),('J',58,63),('K',64,66),('L',68,68),
       ('M',69,75),('N',77,82),('O',84,84),('P',85,85),('Q',86,88),('R',90,93),
       ('S',94,96),('T',97,98),('U',99,99)]

def sec(d):
    for s, a, b in rng:
        if a <= d <= b:
            return s
    return None

df['sectiune'] = df['diviziune'].map(sec)
invalid_codes = df[df['sectiune'].isna()]
valid_codes  = df.dropna(subset=['sectiune'])

g = valid_codes.groupby('sectiune').agg(firme=('id','count'), angajati=('angajati','sum'))
g['pondere_%'] = (100 * g['angajati'] / g['angajati'].sum()).round(1)

print("=" * 55)
print("PAS 1 — Structura pe secțiuni")
print("=" * 55)
print(g)
print("\nCoduri invalide:")
print(invalid_codes[['id', 'denumire', 'caem']].to_string(index=False))

# ============================================================
# PAS 2. Crosswalk 1:n cu alocare
# ============================================================
cw = pd.DataFrame(
    {'cod_intern': ['R1','R2','R3','R4','R4'],
     'nace':       ['10.71','56.10','47.11','62.01','62.02'],
     'pondere':    [1, 1, 1, 0.6, 0.4]})

reg = pd.DataFrame(
    {'firma':      list('ABCDEF'),
     'cod_intern': ['R1','R2','R3','R4','R4','R4'],
     'angajati':   [40, 30, 100, 50, 20, 10]})

m = reg.merge(cw, on='cod_intern')
m['ang_alocati'] = m['angajati'] * m['pondere']

print("\n" + "=" * 55)
print("PAS 2 — Crosswalk cu alocare")
print("=" * 55)
print(m.groupby('nace')['ang_alocati'].sum())

tip = cw.groupby('cod_intern')['nace'].nunique()
print(f"\nTipuri: 1:1 = {(tip == 1).sum()}, 1:n = {(tip > 1).sum()} (R4)")
print(f"Control: {reg['angajati'].sum()} = {m['ang_alocati'].sum():.0f}")

# ============================================================
# PAS 3. Indicatori de calitate
# ============================================================
pub = pd.DataFrame(
    {'lucrare':          [f'L{i}' for i in range(1, 13)],
     'intarziere_zile':  [0,0,0,2,0,0,5,0,0,0,1,0]})

pct            = 100 * (pub['intarziere_zile'] == 0).mean()
rata_raspuns   = 100 * 900 / 2000
completitudine = 100 * valid_codes.shape[0] / len(df)

print("\n" + "=" * 55)
print("PAS 3 — Indicatori de calitate")
print("=" * 55)
print(f"Punctualitate:   {pct:.1f}%")
print(f"Rata de răspuns: {rata_raspuns:.1f}%")
print(f"Completitudine:  {completitudine:.1f}%")

# ============================================================
# EXERCIȚIUL 1. Firme cu cod 0300 și 9900
# ============================================================
print("\n" + "=" * 55)
print("EXERCIȚIUL 1 — Firme cu cod 0300 și 9900")
print("=" * 55)

df1 = pd.concat([
    df,
    pd.DataFrame({'id':[17, 18],
                  'denumire':['Firma 0300', 'Firma 9900'],
                  'activitate':['test', 'test'],
                  'caem':['0300', '9900'],
                  'angajati':[10, 20]})
], ignore_index=True)

df1['caem'] = df1['caem'].str.zfill(4)
df1['diviziune'] = df1['caem'].str[:2].astype(int)
df1['sectiune'] = df1['diviziune'].map(sec)

print(df1[df1['id'].isin([17, 18])][['id','caem','diviziune','sectiune']]
      .to_string(index=False))

# ============================================================
# EXERCIȚIUL 2. Ponderi R4: 0.7 / 0.3
# ============================================================
print("\n" + "=" * 55)
print("EXERCIȚIUL 2 — Ponderi R4: 0.7 / 0.3")
print("=" * 55)

cw2 = cw.copy()
cw2.loc[(cw2['cod_intern'] == 'R4') & (cw2['nace'] == '62.01'), 'pondere'] = 0.7
cw2.loc[(cw2['cod_intern'] == 'R4') & (cw2['nace'] == '62.02'), 'pondere'] = 0.3

m2 = reg.merge(cw2, on='cod_intern')
m2['ang_alocati'] = m2['angajati'] * m2['pondere']

print(m2.groupby('nace')['ang_alocati'].sum())
print(f"\nControl: {reg['angajati'].sum()} = {m2['ang_alocati'].sum():.0f}")

# ============================================================
# EXERCIȚIUL 3. Punctualitate cu toleranță ≤ 1 zi
# ============================================================
print("\n" + "=" * 55)
print("EXERCIȚIUL 3 — Punctualitate (≤ 1 zi = la termen)")
print("=" * 55)

pct_tolerant = 100 * (pub['intarziere_zile'] <= 1).mean()
print(f"Punctualitate (≤ 1 zi): {pct_tolerant:.1f}%")
print(f"(față de {pct:.1f}% fără toleranță)")

# ============================================================
# EXERCIȚIUL 4. Principii CoP vizate de fiecare indicator
# ============================================================
print("\n" + "=" * 55)
print("EXERCIȚIUL 4 — Principii CoP și măsuri propuse")
print("=" * 55)
print("""
1. Punctualitate (75.0%) — principiul 13 CoP (actualitate/timeliness)
   Măsură: notificări automate cu 3 zile înainte de termen.

2. Rata de răspuns (45.0%) — principiile 12 (acuratețe) și 9 (relevanță)
   Măsură: simplificarea chestionarului + canal online de colectare.

3. Completitudine cod CAEM (93.8%) — principiile 12 (acuratețe)
   și 14 (completitudine)
   Măsură: validare automată a codului la introducere + listă de sugestii.
""")

PAS 1 — Structura pe secțiuni
          firme  angajati  pondere_%
sectiune                            
A             2        30        3.0
C             2       107       10.7
F             1       150       14.9
G             1       120       12.0
H             1        35        3.5
I             2        82        8.2
J             2       107       10.7
M             1         6        0.6
P             1        48        4.8
Q             2       319       31.8

Coduri invalide:
 id denumire caem
 16  Firma X 4400

PAS 2 — Crosswalk cu alocare
nace
10.71     40.0
47.11    100.0
56.10     30.0
62.01     48.0
62.02     32.0
Name: ang_alocati, dtype: float64

Tipuri: 1:1 = 3, 1:n = 1 (R4)
Control: 250 = 250

PAS 3 — Indicatori de calitate
Punctualitate:   75.0%
Rata de răspuns: 45.0%
Completitudine:  93.8%

EXERCIȚIUL 1 — Firme cu cod 0300 și 9900
 id caem  diviziune sectiune
 17 0300          3        A
 18 9900         99        U

EXERCIȚIUL 2 — Ponderi R4: 0.7 / 0.3
nace
10.71